In [1]:
# %load_ext autoreload
# %autoreload 2

# from utgen.raggraph.utils import get_node_context
# from utgen.test_generation_crew.crew import TestGenerationCrew

In [7]:
import json

# Open the file in read mode ('r')
with open('../data/tests_data.json', 'r', encoding='utf-8') as f:
    # Load the file content into a Python dictionary
    data = json.load(f)

In [8]:
for k, v in data['tests'].items():
    print(k, v)
    print()

test_normalize_signature_with_basic_function {'name': 'test_normalize_signature_with_basic_function', 'imports': 'import ast\nimport pytest\nfrom utgen.raggraph.utils import normalize_signature', 'code': "def test_normalize_signature_with_basic_function():\n    # Create a simple FunctionDef node\n    node = ast.FunctionDef(\n        name='example',\n        args=ast.arguments(\n            posonlyargs=[],\n            args=[],\n            vararg=None,\n            kwonlyargs=[],\n            kw_defaults=[],\n            kwarg=None,\n            defaults=[]\n        ),\n        body=[],\n        decorator_list=[],\n        returns=None,\n        type_comment=None\n    )\n    \n    result = normalize_signature(node)\n    assert result == 'def example()'"}

test_normalize_signature_with_arguments {'name': 'test_normalize_signature_with_arguments', 'imports': 'import ast\nimport pytest\nfrom utgen.raggraph.utils import normalize_signature', 'code': "def test_normalize_signature_with_argum

In [9]:
import subprocess
import os

def validar_test_individual(import_code, test_code):
    """
    Prova un test en un fitxer temporal. 
    Retorna True si el test passa, False si no.
    """
    temp_filename = "temp_validation.py"
    full_code = f"{import_code}\n\n{test_code}"

    with open(temp_filename, "w") as f:
        f.write(full_code)
    
    # Executem pytest. -q per silenci, --tb=no per no embrutar la consola
    result = subprocess.run(["pytest", temp_filename, "-q", "--tb=no"], capture_output=True)
    
    if os.path.exists(temp_filename):
        os.remove(temp_filename)
        
    return result.returncode == 0

def guardar_i_netejar_tests(tests_valids, desti="../tests/test_generats_llm.py"):
    """
    tests_valids: Llista de tuples [(import, codi), ...]
    """
    if not tests_valids:
        print("No hi ha tests vàlids per guardar.")
        return

    os.makedirs(os.path.dirname(desti), exist_ok=True)

    # 1. SEPAREM EN DOS BLOCS
    bloc_imports = []
    bloc_tests = []
    
    for imp, code in tests_valids:
        bloc_imports.append(imp.strip())
        bloc_tests.append(code.strip())
    
    # 2. CONCATENEM: primer TOTS els imports, després els tests
    contingut_final = "\n".join(bloc_imports) + "\n\n" + "\n\n".join(bloc_tests)

    with open(desti, "w") as f:
        f.write(contingut_final)

    # 2. Passem Ruff per endreçar-ho tot
    print(f"Netejant {desti} amb Ruff...")
    
    # Ordenar imports i eliminar duplicats (isort)
    subprocess.run(["ruff", "check", "--select", "I", "--fix", desti], capture_output=True)
    # # Eliminar variables/imports no usats i altres errors comuns
    subprocess.run(["ruff", "check", "--fix", desti], capture_output=True)
    # # Formatar codi (estil Black)
    subprocess.run(["ruff", "format", desti], capture_output=True)
    
    print(f"✅ Procés finalitzat. Fitxer guardat a: {desti}")

In [10]:
node_id = "utgen/raggraph/walker.py::function::iter_python_files"
node_id = "utgen/raggraph/utils.py::function::normalize_signature"

split_1 = node_id.split('/')
split_2 = split_1[-1].split('::')
split_3 = split_2[0].split('.')

test_func_import = f"\nfrom {split_1[0]}.{split_1[1]}.{split_3[0]} import {split_2[-1]}"
filename = f"test_{split_2[-1]}.py"

In [12]:
tests_que_han_passat = []

for k, v in data['tests'].items():
    name, imp, code = k, v['imports'], v['code']
    # imp = imp + test_func_import
    if validar_test_individual(imp, code):
        print(f"✅ Test `{name}` acceptat")
        tests_que_han_passat.append((imp, code))
    else:
        print(f"❌ Test `{name}` rebutjat")

# Guardem i deixem el fitxer perfecte
guardar_i_netejar_tests(tests_que_han_passat, desti=f"../tests/{filename}")

✅ Test `test_normalize_signature_with_basic_function` acceptat
✅ Test `test_normalize_signature_with_arguments` acceptat
✅ Test `test_normalize_signature_with_return_type` acceptat
✅ Test `test_normalize_signature_with_all_features` acceptat
✅ Test `test_normalize_signature_with_invalid_node` acceptat
Netejant ../tests/test_normalize_signature.py amb Ruff...
✅ Procés finalitzat. Fitxer guardat a: ../tests/test_normalize_signature.py
